# Calorimetry

<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-2-Calorimetry/Figures/Thermogram.png?raw=True"> </center>
</div>


<a target="_blank" href="https://colab.research.google.com/github/pmpatel-udallas/CHE-3131/blob/main/Rotation-2-Calorimetry/Calorimetry.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

### Understanding Temperature Changes During Mixing

When two liquids are combined in a calorimeter to carry out a chemical reaction, the temperature does not change instantaneously even if the reaction itself is rapid. This is largely due to molecular diffusion: it takes time for the substances to fully mix and for the heat released or absorbed by the reaction ($\Delta H_\text{rxn}$) to spread throughout the solution.

In an ideal, perfectly insulated calorimeter, we would observe a flat, constant temperature before the reaction ($T_i$) and again after the reaction has completed ($T_f$). In such a case, calculating the temperature change ($\Delta T$) would be simple:  
$$
\Delta T = T_f - T_i
$$

However, real-world calorimeters are not perfect. Small amounts of heat can be lost or gained from the surroundings, leading to gradual drift in temperature both before and after the reaction. This makes it harder to pinpoint a single moment when the reaction has "fully occurred."

Since there's no single, universally accepted method for identifying this **instantaneous mixing time** ($t_\text{mix}$), we must rely on models and graphical analysis to estimate it. In this activity, you will explore two methods to predict $t_\text{mix}$:
1. $T_{.63R}$ Method
2. Equivalent areas Method


In [ ]:
# @title Overview
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
  </head>

<style>
div.info {
    color: #0056b3;
    background-color: #d9edf7;
    border-left: 5px solid #31708f;
    padding: 0.5em;
    font-size: 1.25em; /* A little larger the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.info ul {
    margin: 0.5em 0; /* Space around the list */
}
div.info li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="info">
    <strong>Questions:</strong>
    <ul>
        <li> What physical processes might cause the temperature to drift before and after the reaction? </li>
        <li>How can we fit a thermogram? </li>
          <ul>
          <li>Can we use analytical functions can we use to fit our data? Will there be uncertainities in the fittings?</li>
          <li>How can fitting lines to the pre- and post-reaction regions help us better estimate the true temperature change?</li>
          <li>What assumptions are we making when we use line fits to model the pre- and post-reaction baselines?</li>
          </ul>
        <li>How do we calculate the instantaneous mixing time (\(t_{mix}\)) from our data?</li>
    </ul>

    <strong>Objectives:</strong>
    <ul>
        <li>Use non-linear numerical fitting techniques to fit experimental data.</li>
        <li>Use Python functions to simplify repetitive tasks.</li>
        <li>Calculate \(\Delta H_{rxn}\) using calorimetric techniques.</li>
    </ul>
</div>

In [ ]:
# @title Hit the play button to import packages
%%capture
# Import standard packages
import numpy as np # Import numerical analysis
import os,sys,re # Import regex
import pandas as pd # DataFrame analysis

#Spectra fitting - scipy
from scipy import interpolate
from scipy import integrate
import scipy.optimize
from scipy import stats
from scipy.stats import skewnorm
from scipy.integrate import quad

#Sliders
import ipywidgets as widgets

# Plotting
import matplotlib
import matplotlib.cm as cm
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 16})
matplotlib.rcParams.update({'font.family': 'Sans'})
matplotlib.rcParams.update({'mathtext.fontset' : 'custom'})

# Insert a progress bar to show the progress of the script
!jupyter nbextension enable --py widgetsnbextension
from tqdm.notebook import tqdm, tnrange, trange

%pip install lmfit
from lmfit import Model

#Quizzes and Animations
from IPython.display import display, HTML, clear_output
import time

%pip install "jupyterquiz"
from jupyterquiz import display_quiz

#Import quizzes and rotation1.json from GitHub
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-2-Calorimetry/rotation2.json
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-2-Calorimetry/t63_interactive.py
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-2-Calorimetry/equiv_interactive.py
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/Rotation-2-Calorimetry/Sample-Data/Calorimetry_SampleData.xlsx
!wget https://raw.githubusercontent.com/pmpatel-udallas/CHE-3131/main/quiz_utils.py

import quiz_utils


In [ ]:
# @title Hit the play button to link your Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


## 1. Calculate $\Delta$T

Typically, in a calorimetry experiment $\Delta T$ must be determined with care for several reasons. The chemical reaction takes place in a finite volume ($\sim$120 mL for this experiment), so the temperature of the surroundings is coming to equilibrium as the reaction proceeds to completion. Thus, there may be a temporary non-uniformity in temperature while the temperature is monitored in only one location in the solution. Stirring greatly helps maintain uniformity of temperature, but in the crucial first few fractions of a second after the reactants are mixed, the instantaneous change in temperature may not be detected. But, stirring may also introduce an endothermic “leak” by imparting some frictional heat into the surroundings. Additionally, the calorimeter may not be fully insulated from its surroundings, introducing an exothermic “leak” as the temperature of the water in the Dewar flask exceeds the temperature outside the flask.

In the apparatus used for this laboratory, the Dewar flask is very well insulated but as soon as the mixture reaches a peak temperature you may see it begin to return slowly to ambient temperature. These issues do not pose an insurmountable barrier to getting the right $\Delta T$, but some interpretation of the “thermogram”, the temperature vs. time trace, is required.

<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-2-Calorimetry/Figures/SolutionCalorimeter.png?raw=True"> </center>
</div>


## 1.1 What is contained within a thermogram?

In [ ]:
# @title ## Exercise: Analyze Interactive Functions
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<p> Analyze the interactive graphs. Think about the following for each method:
<ul>
<li> How does changing x\(_1\), x\(_2\), and x\(_3\) affect \(\Delta T\)? </li>
<li>Does your chosen \( t_{mix} \) minimize residuals or visual error between model and data?</li>
<li>How would using an analytical function help in this analysis?</li>
<li>What does your chosen \( t_{mix} \) reflect in each interactive plot below?</li>
</ul>
</p>
</div>

In [ ]:
# @title Analyze this interactive graph for the $T_{.63R}$ method
%run t63_interactive.py

interactive(children=(FloatSlider(value=91.0, continuous_update=False, description='x1:', max=180.0, step=0.36…

<Figure size 640x480 with 0 Axes>

In [ ]:
# @title Concept Check
quiz_utils.json_to_quiz('rotation2.json','lab-2-2')

<IPython.core.display.Javascript object>

In [ ]:
# @title **Question 1**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
</style>

<div class="gold-note">

    <p>What features does the interactive thermogram plot help you explore,
      and how do they relate to determining the instantaneous mixing time in a calorimetry experiment?</p>
</div>


---

**Answer goes here**

---

In [ ]:
# @title Analyze this interactive graph for the equivalent areas method
%run equiv_interactive.py

interactive(children=(FloatSlider(value=85.0, continuous_update=False, description='x1:', max=130.0, min=70.0,…

<Figure size 640x480 with 0 Axes>

In [ ]:
# @title Concept Check
quiz_utils.json_to_quiz('rotation2.json','lab-2-3')

<IPython.core.display.Javascript object>

In [ ]:
# @title **Question 2**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
</style>

<div class="gold-note">

    <p>What are your initial thoughts of both methods? </p>
    <p>Are the results for \(\Delta T\) consistent between both methods?</p>
    <p>Is there a way to optimize finding \(t_{mix}\) within each method? What kind of algorithms might help in solving this? </p>
  <p>
</div>

---

**Answer goes here**

---

### **Procedure**
1. We need to fit linear lines for pre- and post-reaction.
2. We need to then create a fitting function (either analytical or interpolation) to represent the thermogram.
3. We need to find a way to automate the calculation of $\Delta T$.
  *   We can create a tool that can minimize the difference between $t_{mix}$ and $T_{63R}$.
  *   A separate tool can minimize the difference between the areas.between two curves (linear line and thermogram) to determine $t_{mix}$.
4. Once $\Delta T$ is calculated, we can calculate $\Delta H_{rxn}$.
5. We need to account for propagation of errors.

## 1.2. Import Data

Remember that for all these analyses, we need to first import our data.

In [ ]:
# Import your individual thermograms
# Use index_col='Time' (or whatever you labeled that axis) so that you can filter by time rather than row number


## 1.3 Define Fitting Functions

We are going to explore two sigmoid functions (arctangent, logistic) to fit the data. We can also explore an interpolation function.

In [ ]:
# @title Concept Check
quiz_utils.json_to_quiz('rotation2.json',"lab-2-1")

<IPython.core.display.Javascript object>

In [ ]:
# @title Exercise: Defining Fitting Functions
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>

You've identified the generalized form of a linear function and arctangent function in the concept check above.

A valid generalized form of a logistic function is
$$
f(x) = \dfrac{L}{1+e^{-k(x-x_0)}}+B
$$

<p> <b> Code in all three generalized formulas into the defined Python functions below.</b> All the variables are already included in the function. </p>

</div>

In [ ]:
# Define the fitting functions for the thermogram
def line(x, m, b):
  ''' Define a function for a line '''
  y =
  return y

def logistic(x, L ,x0, k, B):
  ''' Define a logistic function '''
  y =
  return y

def ARCTAN(x, a, b, c, d):
  y =
  return y

# Interpolation function
def interp(x,y):
  '''x=time, y=temperature'''
  f = interpolate.interp1d(x,y)
  return f

**The figure below shows an example of how the code will be used with the specific variables defined to calculate $t_{mix}$.**

**Use the code blocks below to calculate $\Delta T$ with each method.**

<div>
  <center> <img height="500" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-2-Calorimetry/Figures/Thermogram_VariableDiagram.png?raw=True"> </center>
</div>

In [ ]:
# @title Exercise: Establish fitting parameters
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
You need to identify which values will be used to fit the linear lines:
<ul>
  <li> \(x_1\) (<code>x1</code>) -- this is the point right before the reaction begins</li>
  <li> \(x_3\) (<code>x3</code>) -- this is where the thermogram approaches linearity post-reaction</li>
  <li> Plot the bounds (0 to x1) and (x3 to end) of the thermogram as different colors so you can visually see where your data is linear.</li>
</ul>
<p> Your resulting graph will look similar to the interactive graphs from earlier.</p>

</div>

In [ ]:
# @title Note
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

<p> <strong> Note: </strong> Since you imported your data with time being the <code>index_col</code> option, use <code>df.loc[]</code>
to slice the data based on time stamps rather than data point number. </p>
</div>

<style>

In [ ]:
# Put your code here to determine the bounds for pre- and post-reaction lines
# Create a plot and change the bounds to determine linearity




In [ ]:
# Calculate the linear lines before and after
xdata=
ydata=

# Define the best fit lines using scipy.stats.linregress
Ti1=
Tf1=

In [ ]:
# Calculate the analytical logistic function with best fit parameters
# Prints out the statistics of the fit, including uncertainties

xdata=
ydata=

# Fit the data with initial conditions using LMFIT Model
gmodel1 = Model(logistic)

# Pick initial conditions for ydata, xdata, and fitting parameters
# x0 and B should be changed based on your information
# x0 is the time you hit the plunger
# B is the temperature
result1 = gmodel1.fit(ydata, x=xdata, L=0.6, x0=95, k=0.6, B=21.1)

# Return the best fit parameters
L=result1.params['L'].value
x0=result1.params['x0'].value
k=result1.params['k'].value
B=result1.params['B'].value

# Print the standard statistics of the fitting
print(result1.fit_report())

# Print the confidence interval values for each parameter
print(result1.ci_report())

In [ ]:
# Calculate the analytical arctangent function with best fit parameters
# Prints out the statistics of the fit, including uncertainties

xdata=
ydata=

# Fit the data with initial conditions
gmodel2 =

# Pick initial conditions for ydata, xdata, and fitting parameters
result2 =

# Return the best fit parameters


# Print the standard statistics of the fitting
print(result2.fit_report())

# Print the confidence interval values for each parameter
print(result2.ci_report())

In [ ]:
# @title Note
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

    <p><strong>What about using multiple functions?</strong></p>

 <p> After lab, try a composite model that has the form (logistic + arctan) and one that
  is a linear combination (c1*logistic + c2*arctan) and fit c1 and c2. </p>
 <p> You can try any alternative you think is worth your time exploring,
  some examples are: </p>
  <ul>
    <li> line + exponential
    <li> line + logistic
    <li> logistic + arctan
    <li> line + arctan
    <li> logistic + exponential
</ul>
See if any of these can improve the \(r^2\) value if you want to explore potentially better fitting functions.

<p>
For a deeper explanation of LMFIT and the code for composite models, follow this link to read about
<a href="https://lmfit.github.io/lmfit-py/model.html" target="_blank">LMFIT Models</a>.
</p>

</div>

In [ ]:
# @title Exercise: Creating Representative Figures
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
Create a representative figure of this method that displays the following:
<ul>
  <li> the thermogram </li>
  <li> the best fit lines </li>
  <li> the sigmoid fitting function </li>
  <li> the \(T_{63R}\) line </li>
  <li> Include axes labels and units but no plot title</li>
</ul>
This figure will be the base of what goes in the lab report to model the various components of this analysis.
</div>

In [ ]:
# You can use the code from the pre-lab assignment to plot the best fit model





## 1.4 T$_{.63R}$


In [ ]:
# @title Exercise: Interpret the $T_{63R}$ method
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
\(T_{.63R}\) is the temperature that is 63.2% of \(\Delta T\) above \(T_i\).

$$
	T_{.63R}=T_i + (\Delta T \times 0.632)
$$

<p>\(\Delta T\) is the difference between the extrapolated pre-reaction trend and the extrapolated post-reaction trend on a
vertical line at the equivalent instantaneous mixing time (**t$_{MIX}$**), which intersects the thermogram at \(T_{.63R}\).
This analysis should be done numerically, fitting both the pre-reaction and post reaction data to a straight line.</p>

By minimizing the difference between the intersection between the \(T_{.63R}\) line
(the line that sits at 63% between \(T_i\) and \(T_f\))and the thermogram, \(t_\text{MIX}\),
which intersects the thermogram at \(T_{.63R}\), can be found and can then be extrapolated to solve for \(\Delta T\).

<p> <b>Interpret the Python function called diff_T63. What does it do? Fill in the blanks in the function to make it run correctly.</b></p>
<p> Then, in the two cells below, create new versions of diff_T63 that will run the regression on other functions (arctangent, logistic, interpolation). </p>
</div>

In [ ]:
def diff_T63(x2):

  # Define the points along the best fit lines and the thermogram
  # Use the line functions for yi and yf
  yi=
  yf=

  # Define which best fit function you are using for y
  y=

  # Calculate the ratio between the y values of the best fit lines
  # and the analytical function for the thermogram (You will code this)
  dy=

  # Store the iteration into a global list
  diff1_data.append((x2, y))

  # Return the absolute value of the difference between dy and 0.632 (63.2%)
  return

In [ ]:
# Practice cell to test different parts of the calculation before putting it in the function





In [ ]:
# Try running different values of diff_T63 to see how it works






In [ ]:
# @title Note
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

<p> <strong> Note: </strong> </p>

<code>scipy.optimize.minimize_scalar</code> is a <b>Python function used to find the minimum value of a single-variable function</b>
— that is, it tries to solve:

$$
\min_x f(x)
$$

Where \(f(x)\) is a function that returns a number (a scalar), and you want to find the value of \(x\) that makes \(f(x)\) as small as possible.

Use <code>minimize_scalar</code> when:
<ul>
<li> You have a <b>function of one variable</b> (like a cost, energy, error, etc.).</li>
<li> You want to <b>find the input \(x\) that gives the smallest output</b>.</li>
<li> You don’t need to compute gradients or use multi-variable optimization.</li>
</ul>
</div>

<style>

In [ ]:
# Find where T63R intersects the thermogram through an optimization procedure.

# Set the bounds to those of the curve of the thermogram (x1 and x3).
x1=
x3=

# Define the global list diff1_data to store the iterations
diff1_data=[]

# Run the minimize scalar optimization procedure
g=scipy.optimize.minimize_scalar(diff_T63,bounds=(x1,x3))
print("tmix T63R:",g.x)

tmix T63R: 96.35193685222961


In [ ]:
# @title Note
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

<p> <strong> Note: </strong> </p>
<p> The animation below uses <code>result1.best_fit</code> to represent the thermogram.
  Make sure whichever fitting function you are using from LMFIT is labeled as <code>result1</code>.</p>
 The thermogram data is labeled as <code>xdata</code> and <code>ydata</code>.</p>
</div>

<style>

### Visualize the optimization process

The below cell will play an animation of the optimization process. Press the play button to run the animation.

In [ ]:
# @title Hit the play button

from IPython.display import display, clear_output
import time
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.cm as cm

# Extract x and function values from the stored data
iterations = [data[0] for data in diff1_data]
function_values = [data[1] for data in diff1_data]

# Create a figure and axes
fig, ax = plt.subplots(figsize=(7, 5))

# Plot the thermogram and the best fit logistic curve
ax.plot(xdata, ydata, 'ko', alpha=0.05, markersize=4)
ax.plot(xdata, result1.best_fit, '-', label='Best Fit', color='k', lw=1)

# Set axes labels and title
ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature ($^\degree$C)')
ax.grid(True)

# Create a scatter plot object for the iterations
# Initialize with empty data but set up the colormap
scatter = ax.scatter([], [], c=[], s=50, cmap='coolwarm', vmin=0, vmax=len(iterations)-1)

# Display the initial plot
display(fig)

# Iterate through the stored data and update the scatter plot
for i in range(len(iterations)):
    # Update the scatter plot data
    scatter.set_offsets(np.c_[iterations[:i+1], function_values[:i+1]])
    # Assign colors based on the iteration number
    # Use a range from 0 to the current iteration index to map to the colormap
    scatter.set_array(np.arange(i + 1))

    # Clear the previous output and display the updated figure
    clear_output(wait=True)
    display(fig)
    time.sleep(0.25)  # Small delay to visualize the update

# Close the figure after the loop finishes
plt.close(fig)

In [ ]:
# @title **Question 3**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
</style>

<div class="gold-note">

    <p>Put your observations about the animation and the T63R method below.</p>
  <p>
</div>

---


**Answer goes here**

---

## 1.5 Equivalent Areas

Define the integrals and then calculate the difference between the two areas A and B.

<div>
  <center> <img height="400" src="https://github.com/pmpatel-udallas/CHE-3131/blob/main/Rotation-2-Calorimetry/Figures/Thermogram_VariableDiagram.png?raw=True"> </center>
</div>

In [ ]:
# @title Exercise: Interpret the Equivalent Areas method
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>

<p>
In calorimetry experiments, the heat released or absorbed would ideally cause an immediate jump in temperature,
but in real systems, the temperature changes gradually due to mixing and thermal lag.
</p>

<p>
To accurately measure the total heat change, we use the <strong>equivalent areas method</strong>.
This method accounts for the delayed temperature response by comparing the experimental temperature data
to what the temperature would have been without the reaction.
</p>


<p>
The area between the experimental curve and the baselines represents the total temperature change caused by the reaction,
spread out over time. We treat this area as if it came from an <em>instantaneous</em> temperature change (\(t_2\)):
</p>

$$ \int_{t_1}^{t_2} \left[ T_{\text{exp}}(t) - T_i(t) \right] dt
  =
  \int_{t_2}^{t_3} \left[ T_f(t) - T_{\text{exp}}(t) \right] dt
$$

<p>
This gives an <strong>equivalent temperature change</strong> \( \Delta T_{\text{eq}} \),
which can then be used in the usual calorimetry formula:
</p>

<p style="text-align: center;">
  \( q = C_{cal} \cdot \Delta T_{\text{eq}} \)
</p>

<p>
This method works even if the temperature curve is not smooth or has some delay
because it focuses on the <em>total thermal effect</em> and not just the peak temperature.
</p>

<p> <strong> Questions: </strong></p>
<ul>
<li> What does this function do? What are the steps involved? </li>
<li> What are the roles of x1, x2, and x3 in this function?</li>
<li> You will calculate the areas under the respective curves using <code>scipy.integrate</code> functions,
  how will you calculate the area between two curves?</li>
</ul>

</div>

In [ ]:
# The best fit parameters you defined earlier will carry over into these lines
def diff_EqualAreas(x2, x1, x3):
  '''
  x1 is lower intergration limit
  x2 is middle integration limit and will be tmix
  x3 is upper integration limit
  '''

  # Calculate area B (Thermogram Best Fit Curve - Temp Init Line)
  Thermo_init=
  Line_init=scipy.integrate.quad(line, x1, x2, args=(Ti1.slope, Ti1.intercept))[0]

  # Calculate area A (Temp final line - Thermogram Best Fit Curve)
  Thermo_final=
  Line_final=scipy.integrate.quad(line, x2, x3, args=(Tf1.slope, Tf1.intercept))[0]



  # Calculate the areas (You will code this)
  area_A=
  area_B=

  # Once you calculate area_A and area_B, then calculate the absolute value of
  # the difference between areas (You will code this)
  area_diff =

  # Append the values to the global list
  diff2_data.append((x2, logistic(x2,L,x0,k,B)))

  # Return the absolute value of the difference between areas (You will code this)
  return

In [ ]:
# Practice cell to test different parts of the calculation before putting it in the function





In [ ]:
# Try running different values of diff_EqualAreas to see how it works




In [ ]:
# Create a global list to store the data
diff2_data = []

# Define the values for x1 and x3 (integration bounds)
x1 = 90
x3 =

# We will use minimize instead of minimize_scalar to handle multiple arguments (x1 and x3)
# We have to provide an initial guess x0 which is set to x1+0.5 so it can run the visualization code
h = scipy.optimize.minimize(diff_EqualAreas, x0=x1+0.5,args=(x1, x3), bounds=[(x1, x3)])

# Print the output x value for t_mix.
# h.x is an array when using minimize, so h.x[0] calls the value we need
print('tmix area:','%0.5f'%h.x[0])

### Visualize the optimization process

The below cell will play an animation of the optimization process. Press the play button to run the animation.

In [ ]:
# @title Hit the play button

# Extract x and function values from the stored data
# Ensure iterations and function_values are numpy arrays for easier slicing
iterations = np.array([data[0] for data in diff2_data]).flatten() # Flatten the arrays if they contain single-element arrays
function_values = np.array([data[1] for data in diff2_data])

# Create a figure and axes
fig, ax = plt.subplots(figsize=(7, 5))

# Plot the thermogram and the best fit logistic curve once outside the loop
ax.plot(xdata, ydata, 'ko', alpha=0.05, markersize=4, label='Data')
ax.plot(xdata, result1.best_fit, '-', label='Best Fit (Logistic)', color='k', lw=1)


# Set axes labels and title
ax.set_xlabel('Time (s)')
ax.set_ylabel('Temperature ($^\degree$C)')
ax.grid(True)

# Create a scatter plot object for the iterations
# Initialize with empty data but set up the colormap
# Set zorder higher to ensure points are on top
scatter = ax.scatter([], [], c=[], s=50, cmap='coolwarm', vmin=0, vmax=len(iterations)-1, zorder=10) # Increased zorder


# Initialize lists to store lines and collections to be removed
lines_to_remove = []
collections_to_remove = []

# Iterate through the stored data and update the scatter plot
for i in range(len(iterations)):
    # Remove lines and collections from the previous iteration
    for line_obj in lines_to_remove:
        line_obj.remove()
    for collection_obj in collections_to_remove:
        collection_obj.remove()

    # Clear the lists for the current iteration
    lines_to_remove = []
    collections_to_remove = []

    # Update the scatter plot data with cumulative points up to the current iteration
    scatter.set_offsets(np.c_[iterations[:i+1], function_values[:i+1]])
    # Assign colors based on the iteration number
    scatter.set_array(np.arange(i + 1))

    x2_tmp = iterations[i]

    # Define integration bounds based on the current x2_tmp
    xlo2_mask = (xdata >= x1) & (xdata <= x2_tmp)
    xlo2 = xdata[xlo2_mask]

    xhi2_mask = (xdata >= x2_tmp) & (xdata <= x3)
    xhi2 = xdata[xhi2_mask]

    # Plot the best fit lines (assuming Ti1 and Tf1 are defined) in each iteration
    line1, = ax.plot(xdata, line(xdata, Ti1.slope, Ti1.intercept), 'C1--', label='pre-reaction line')
    line2, = ax.plot(xdata, line(xdata, Tf1.slope, Tf1.intercept), 'C2--', label='post-reaction line')
    # Add lines to the remove list
    lines_to_remove.extend([line1, line2])

    # Plot the filled areas in each iteration
    fill1 = ax.fill_between(xlo2, line(xlo2, Ti1.slope, Ti1.intercept),
                     logistic(xlo2, L, x0, k, B), color='C0', alpha=0.2)
    fill2 = ax.fill_between(xhi2, line(xhi2, Tf1.slope, Tf1.intercept),
                     logistic(xhi2, L, x0, k, B), color='C0', alpha=0.2)

    # Add collections to the remove list
    collections_to_remove.extend([fill1, fill2])


    # Ensure the limits are consistent if needed
    ax.set_xlim(xdata.min(), xdata.max())
    ax.set_ylim(ydata.min() - 0.05, ydata.max() + 0.05)

    # Update the legend to include the dynamically added lines/fills if needed (optional)
    # ax.legend()


    # Clear the previous output and display the updated figure
    clear_output(wait=True)
    display(fig)
    time.sleep(0.25)  # Small delay to visualize the update

# Close the figure after the loop finishes
plt.close(fig)

In [ ]:
# @title **Question 4**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
</style>

<div class="gold-note">
    <p>Put your observations about the animation and the equivalent areas method below.</p>
</div>

---


**Answer goes here**

---

## 1.6 Calculate $\Delta T$ from both methods

In [ ]:
# Calculate Delta T with each method
deltaT_T63=
deltaT_EQA=

print('ΔT for T63R:','%0.5f'%deltaT_T63)
print('ΔT for equivalent areas:','%0.5f'%deltaT_EQA)

In [ ]:
# @title **Question 5**
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.gold-note {
    color: #796748; /* Dark gold for text */
    background-color: #F7F0E2; /* Light cream/gold background */
    border-left: 5px solid #D8B266; /* Medium gold accent border */
    padding: 0.5em;
    font-size: 1.25em;
    line-height: 1.5;
    font-family: Arial, sans-serif;
}
</style>

<div class="gold-note">
    <p>What are your initial thoughts on the calculated \( \Delta T\) between the two methods?
      Make a note in your physical lab notebook or here in the digital notebook.</p>
</div>

---


**Answer goes here**

---

## 1.7 Calculate $\Delta H$


In [ ]:
# @title Note
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.purple-box {
    color: #4b0082; /* Indigo for text */
    background-color: #f3e5f5; /* Light lavender background */
    border-left: 5px solid #7b1fa2; /* Medium purple border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
    font-family: Arial, sans-serif; /* Clean, modern font */
}
div.purple-box ul {
    margin: 0.5em 0; /* Space around the list */
}
div.purple-box li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="purple-box">

<p> <strong> Note: </strong> </p>

<p> Use a numpy array to calculate the values using the computed \( \Delta T\) from both methods at the same time.

<style>

In [ ]:
mass_HCl=
mass_NaOH=
mass_total=
moles_NaOH=
C_cal=89.44+4.184*mass_total
q_sys=
deltaH=

print('ΔH',deltaH)

## 1.8 Repeat this analysis for every trial

In [ ]:
# @title Exercise: Repeating a procedure
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<p> Now that you have the results for \( \Delta H \) for one trial, do the same analysis for the other trials.<p>
<p> Think about how you can combine the code blocks you did above into a new block below to calculate \( \Delta H \) for  <b> each </b> trial.</p>
<p> This may take on the form of copy+paste the relevant cells from above here and change the values based on each trial
  ...or perhaps one function that combines it all? </p>
</div>

In [ ]:
# Identify the bounds for the linear lines pre- and post-reaction
# Plot the values as we did for the first trial to see which values





In [ ]:
# Put your code here





In [ ]:
# Don't forget about error propagation


# Calculate the base values
mass_HCl=
mass_NaOH=
mass_total=
moles_NaOH=
C_cal=89.44+4.184*mass_total
q_sys=
deltaH=


# Propagate the errors
e_mass_HCl =
e_mass_NaOH =
e_mass_total =
e_conc_NaOH =
e_mols_NaOH =
e_C_cal =
e_DeltaT =
e_q_sys =
e_DeltaH =

print(f'mol_NaOH (1σ): {e_mols_NaOH:.0e}')
print(f'C_cal (1σ): {e_C_cal:0.2f}')
print(f'q_sys (1σ): {e_q_sys:0.0f}')
print(f'ΔH (1σ): {e_DeltaH:0.1f}')

## 1.9 Calculate $\Delta H_{neut}$



In [ ]:
# @title Exercise: Plot the $\Delta H_{rxn}$ vs moles NaOH
%%html
 <head>
    <!-- See http://docs.mathjax.org/en/latest/web/start.html#using-mathjax-from-a-content-delivery-network-cdn -->
    <script type="text/javascript" id="MathJax-script" async
        src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js">
    </script>
</head>

<style>
div.orange-alert {
    color: #854f00; /* Darker shade of orange for text */
    background-color: #ffe6cc; /* Light orange background */
    border-left: 5px solid #ff9933; /* Bright orange border */
    padding: 0.5em;
    font-size: 1.25em; /* Matches the surrounding text size */
    line-height: 1.5; /* Ensures readability */
}
div.orange-alert ul {
    margin: 0.5em 0; /* Space around the list */
}
div.orange-alert li {
    margin-bottom: 0.5em; /* Space between list items */
}
</style>

<div class="orange-alert">
<p> <strong> Exercise: </strong> </p>
<p> Plot the \(\Delta H_{rxn}\) vs moles NaOH for determining \(\Delta H_{neut}\) (heat of neutralization).
This will use the data from all trials.</p>

<p> Based on the concept of infinite dilution, the heat of neutralization is determined as the intercept of this plot.</p>
</div>

In [ ]:
# Create the plot here




